# Breakthrough v1: 보수적 최적화 (Robust & Conservative)

이 노트북은 점수가 계속 나빠지는 원인을 '과한 보정'으로 진단하고, **최소한의 핵심 피처와 강력한 규제**를 통해 68점 벽을 깨는 것을 목표로 합니다.

**수정 전략:**
1. **Strong Regularization:** XGBoost에 L1, L2 규제를 강하게 걸어 오버피팅 방지.
2. **Clean Dinner Data:** 석식계가 0인 데이터를 제거하고 학습하여 모델의 왜곡 방지.
3. **Median-based Correction:** 평균이 아닌 중앙값을 사용하여 이상치에 강한 보정 적용.
4. **Feature Simplification:** 노이즈가 심한 날씨/상세메뉴 피처를 제거하고 운영 지표에만 집중.

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# 경로 자동 탐색 (CY2 환경)
def find_path(cands): 
    for p in cands: 
        if os.path.exists(p): return p
    return None

train_path = find_path([r"C:\Users\CY2\github\SecondProject\data\train_median.csv", r"./data/train_median.csv"])
test_path = find_path([r"C:\Users\CY2\github\SecondProject\data\test.csv", r"./data/test.csv"])
sub_path = find_path([r"C:\Users\CY2\github\SecondProject\data\sample_submission.csv", r"./data/sample_submission.csv"])

train = pd.read_csv(train_path, encoding="utf-8-sig")
test = pd.read_csv(test_path, encoding="utf-8-sig")
sample_submission = pd.read_csv(sub_path, encoding="utf-8-sig")

train["일자"] = pd.to_datetime(train["일자"])
test["일자"] = pd.to_datetime(test["일자"])

In [ ]:
# 최소한의 핵심 피처 생성
def add_simple_features(df):
    df = df.copy().sort_values("일자").reset_index(drop=True)
    df["월"] = df["일자"].dt.month
    df["요일"] = df["일자"].dt.weekday
    # 식사가능자수 (가장 중요한 피처)
    df["식사가능자수"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - df["현본사소속재택근무자수"]).clip(lower=1)
    
    # 운영상 유의미한 플래그만 남김
    df["is_fri"] = (df["요일"] == 4).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["overtime_log"] = np.log1p(df["본사시간외근무명령서승인건수"])
    
    return df

train = add_simple_features(train)
test = add_simple_features(test)
train["중식참여율"] = train["중식계"] / train["식사가능자수"]
train["석식참여율"] = train["석식계"] / train["식사가능자수"]

In [ ]:
# [데이터 정제] 석식이 0인 날은 학습에서 제외 (모델의 편향 방지)
train_lunch = train.copy()
train_dinner = train[train["석식계"] > 0].copy()

# 피처 목록 (불필요한 노이즈 제거)
features = ["월", "요일", "식사가능자수", "본사출장자수", "본사시간외근무명령서승인건수", "is_fri", "is_wed", "overtime_log"]

# [강력한 규제 적용] 모델을 보수적으로 설정
xgb_params = {
    "n_estimators": 1000,
    "learning_rate": 0.02, # 더 천천히 학습
    "max_depth": 4,        # 복잡도 감소
    "reg_lambda": 10,      # L2 규제 강화
    "reg_alpha": 1,        # L1 규제 추가
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

model_l = XGBRegressor(**xgb_params)
model_d = XGBRegressor(**xgb_params)

model_l.fit(train_lunch[features], train_lunch["중식참여율"])
model_d.fit(train_dinner[features], train_dinner["석식참여율"])

train_lunch["l_pred"] = model_l.predict(train_lunch[features])
train_dinner["d_pred"] = model_d.predict(train_dinner[features])

train_lunch["l_res"] = train_lunch["중식참여율"] - train_lunch["l_pred"]
train_dinner["d_res"] = train_dinner["석식참여율"] - train_dinner["d_pred"]

In [ ]:
# [중앙값 기반 미세 보정] 평균보다 이상치에 강함
l_weekday_adj = train_lunch.groupby("요일")["l_res"].median() * 0.5 # 영향력을 50%로 제한
d_weekday_adj = train_dinner.groupby("요일")["d_res"].median() * 0.5

pred_l_ratio = model_l.predict(test[features]) + test["요일"].map(l_weekday_adj).fillna(0)
pred_d_ratio = model_d.predict(test[features]) + test["요일"].map(d_weekday_adj).fillna(0)

pred_l = pred_l_ratio * test["식사가능자수"]
pred_d = pred_d_ratio * test["식사가능자수"]

submission = sample_submission.copy()
submission[submission.columns[1]] = np.clip(pred_l.values, 0, None)
submission[submission.columns[2]] = np.clip(pred_d.values, 0, None)

submission.to_csv("breakthrough_v1.csv", index=False, encoding="utf-8-sig")
print("저장 완료: breakthrough_v1.csv")